In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import sys
sys.path.append('/kaggle/input/entransformer-datasets')
import torch
from torch.utils.data import DataLoader

In [2]:
%%capture

!pip install neuralforecast
!pip install gluonts
!pip install lightning

### Deterministic Seed Setting



In [3]:

import os
import random
import numpy as np
import torch
import cv2
from transformers import set_seed
from datasets import disable_progress_bar
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# optional for debugging only:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

class Deterministic:
    def __init__(self):
        pass

    def init_all(self, seed=0, disable_list=['cuda_block']):
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
        if 'cuda_block' not in disable_list: # stuck when train deberta
            os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
        os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        if 'torch_deter_algo' not in disable_list: # consumn more gpu sometimes
            torch.use_deterministic_algorithms(True, warn_only=True)
        set_seed(seed)
        cv2.setRNGSeed(seed)
        disable_progress_bar()

deterministic = Deterministic()

In [4]:
import os
print(os.getcwd())

c:\Users\Anusha\engression\engression-ts\engressionts\experiments\solar\nf-solar


In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]   # .../engression-ts
sys.path.insert(0, str(project_root))

In [6]:
# import traceback

# print("=" * 80)
# print("Testing EnBEATS import...")
# print("=" * 80)

# try:
#     from engressionts.models.darts.nbeats import EnBEATSModel
#     print("\n✅ SUCCESS")
#     print(EnBEATSModel)

# except Exception as e:
#     print("\n❌ IMPORT FAILED")
#     print(f"\nException type: {type(e).__name__}")
#     print(f"\nException message:\n{e}")

#     print("\nFull traceback:")
#     traceback.print_exc()

# print("\n" + "=" * 80)

In [7]:
# import traceback

# print("=" * 80)
# print("Testing EnHiTS import...")
# print("=" * 80)

# try:
#     from engressionts.models.darts.nhits import EnHiTSModel
#     print("\n✅ SUCCESS")
#     print(EnHiTSModel)

# except Exception as e:
#     print("\n❌ IMPORT FAILED")
#     print(f"\nException type: {type(e).__name__}")
#     print(f"\nException message:\n{e}")

#     print("\nFull traceback:")
#     traceback.print_exc()

# print("\n" + "=" * 80)

In [8]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
PyTorch CUDA version: 12.8
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [9]:
import os

print("CUDA_HOME:", os.environ.get("CUDA_HOME"))
print("CUDA_PATH:", os.environ.get("CUDA_PATH"))

CUDA_HOME: None
CUDA_PATH: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.8


In [10]:
import sys
from pathlib import Path

# Project root of the engression-ts repository
project_root = Path(r"C:\Users\Anusha\engression\engression-ts")

# Add repository root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Project exists:", project_root.exists())
print("engressionts exists:", (project_root / "engressionts").exists())

Project root: C:\Users\Anusha\engression\engression-ts
Project exists: True
engressionts exists: True


### Importing Libraries

In [11]:
import torch
import torch.nn as nn
import pandas as pd
from typing import Tuple, Optional

# NeuralForecast
from neuralforecast import NeuralForecast

# Change ONLY this import when testing a different
# NeuralForecast Engression model.
#
# Examples:
# EnxLSTM          -> enxlstm
# EnPatchTST       -> enpatchtst
# EnAutoformer     -> enautoformer
# EnInformer       -> eninformer
# EnFEDformer      -> enfedformer
# EniTransformer   -> enitransformer
# EnTimeXer        -> entimexer
# EnTimesNet       -> entimesnet
# EnTSMixerx       -> entsmixerx
# EnVanillaTransformer -> envanillatransformer
# EnMLP            -> enmlp
# EnMLPMultivariate -> enmlpmultivariate
# EnKAN            -> enkan
# EnBiTCN          -> enbitcn
# EnXLinear        -> enxlinear
#
# Keep the rest of the notebook unchanged when switching models.

from engressionts.models.neuralforecast.enxlstm import EnxLSTM

W0809 12:58:19.638000 36688 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


## Data Specific Params

In [12]:
# Forecast horizon: number of future timesteps to predict.
PRED_LEN = 24

# LAGS are not required for the NeuralForecast models.
# NeuralForecast uses input_size as the historical context length.

## Importing Data and Formatting

In [13]:
import numpy as np
import pandas as pd

from gluonts.dataset.repository.datasets import get_dataset


def gluonts_to_neuralforecast(dataset):
    """
    Convert a GluonTS dataset to the long-format DataFrame
    expected by NeuralForecast.

    Output columns:
        unique_id -> identifies each Solar node
        ds        -> timestamp
        y         -> observed value
    """
    rows = []

    for i, entry in enumerate(dataset):
        start = (
            entry["start"].to_timestamp()
            if hasattr(entry["start"], "to_timestamp")
            else pd.Timestamp(entry["start"])
        )

        target = np.asarray(entry["target"], dtype=np.float32)

        idx = pd.date_range(
            start=start,
            periods=len(target),
            freq=entry["start"].freqstr,
        )

        for timestamp, value in zip(idx, target):
            rows.append({
                "unique_id": f"node_{i}",
                "ds": timestamp,
                "y": float(value),
            })

    return pd.DataFrame(rows)


# Change the dataset name here if testing a different GluonTS dataset.
ds = get_dataset("solar_nips", regenerate=False)

# Convert train and test data to NeuralForecast format.
train_df = gluonts_to_neuralforecast(ds.train)
test_df = gluonts_to_neuralforecast(ds.test)

# Basic checks
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of nodes:", train_df["unique_id"].nunique())
print(train_df.head())

c:\Users\Anusha\engression\engression-ts\.venv\Lib\site-packages\gluonts\json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


Train shape: (960233, 3)
Test shape: (6813695, 3)
Number of nodes: 137
  unique_id                  ds    y
0    node_0 2006-01-01 00:00:00  0.0
1    node_0 2006-01-01 01:00:00  0.0
2    node_0 2006-01-01 02:00:00  0.0
3    node_0 2006-01-01 03:00:00  0.0
4    node_0 2006-01-01 04:00:00  0.0


In [15]:
# from darts.dataprocessing.transformers import Scaler

# y_scaler = Scaler()                     # StandardScaler-like wrapper
# train_y_sc = y_scaler.fit_transform(train_ts)

# import numpy as np
# from darts import TimeSeries, concatenate

# # Change the lag values if you want to use a different lag configuration.
# def lag_covs_from_scaled_target(ts_sc: TimeSeries, lags=(1,24,168)) -> TimeSeries:
#     shifted = []
#     for L in lags:
#         s = ts_sc.shift(L).with_columns_renamed(
#             ts_sc.components, [f"{c}_lag{L}" for c in ts_sc.components]
#         )
#         shifted.append(s)

#     common = shifted[0]
#     for s in shifted[1:]:
#         common = common.slice_intersect(s)
#     shifted = [s.slice_intersect(common) for s in shifted]
#     return concatenate(shifted, axis=1)

# def fourier_from_index(idx) -> TimeSeries:
#     hour = idx.hour.to_numpy()
#     dow  = idx.dayofweek.to_numpy()
#     X = np.vstack([
#         np.sin(2*np.pi*hour/24.0),
#         np.cos(2*np.pi*hour/24.0),
#         np.sin(2*np.pi*dow/7.0),
#         np.cos(2*np.pi*dow/7.0),
#     ]).T
#     return TimeSeries.from_times_and_values(idx, X, columns=["h_sin","h_cos","dow_sin","dow_cos"])

# def dim_indicator_norm(idx, D: int) -> TimeSeries:
#     v = (np.arange(D, dtype=np.float32) / (D-1)).astype(np.float32)  # 0..1
#     X = np.tile(v, (len(idx), 1))
#     cols = [f"dim_id_{i}" for i in range(D)]
#     return TimeSeries.from_times_and_values(idx, X, columns=cols)

# # Builds past covariates using lag features, node IDs, and time features.
# def build_past_covs_552(ts_sc: TimeSeries, lags=(1,24,168)) -> TimeSeries:
#     lag_covs = lag_covs_from_scaled_target(ts_sc, lags)
#     idx = lag_covs.time_index
#     time_covs = fourier_from_index(idx)
#     dim_covs  = dim_indicator_norm(idx, ts_sc.width)
#     return concatenate([lag_covs, dim_covs, time_covs], axis=1)


# def ts_upto(ts, end_time):
#     # Compatible with multiple Darts versions.
#     if hasattr(ts, "slice_end"):
#         return ts.slice_end(end_time)
#     if hasattr(ts, "drop_after"):
#         return ts.drop_after(end_time)
#     if hasattr(ts, "split_after"):
#         return ts.split_after(end_time)[0]
#     return ts.slice(ts.start_time(), end_time)

In [16]:
# NeuralForecast uses the long-format DataFrame:
# unique_id | ds | y
#
# train_df and test_df were already created in the previous cell.
# No Darts-style wide DataFrame or test_windows are needed here.

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of nodes:", train_df["unique_id"].nunique())

print("\nTrain:")
print(train_df.head())

print("\nTest:")
print(test_df.head())

Train shape: (960233, 3)
Test shape: (6813695, 3)
Number of nodes: 137

Train:
  unique_id                  ds    y
0    node_0 2006-01-01 00:00:00  0.0
1    node_0 2006-01-01 01:00:00  0.0
2    node_0 2006-01-01 02:00:00  0.0
3    node_0 2006-01-01 03:00:00  0.0
4    node_0 2006-01-01 04:00:00  0.0

Test:
  unique_id                  ds    y
0    node_0 2006-01-01 00:00:00  0.0
1    node_0 2006-01-01 01:00:00  0.0
2    node_0 2006-01-01 02:00:00  0.0
3    node_0 2006-01-01 03:00:00  0.0
4    node_0 2006-01-01 04:00:00  0.0


In [17]:
print(train_df.shape)

(960233, 3)


### Metric

In [18]:
import numpy as np
import pandas as pd


def crps(preds, targets, quantiles=(np.arange(20) / 20.0)[1:]):
    """
    Calculate normalized quantile-based CRPS.

    Parameters
    ----------
    preds : np.ndarray
        Shape:
            (B, N, T)
        or
            (B, N, T, D)

        B = forecast windows
        N = stochastic samples
        T = forecast horizon
        D = dimensions/nodes when applicable

    targets : np.ndarray
        Shape:
            (B, T)
        or
            (B, T, D)

    Important:
        This follows the same CRPS formulation used in the
        Darts/Engression evaluation so that we can later make
        the NF evaluation comparable.
    """

    # Convert stochastic samples into the requested quantiles.
    #
    # Example:
    # 100 stochastic samples
    #       ↓
    # 19 quantiles
    #
    # These quantiles are calculated along the sample dimension N.
    x = np.quantile(
        preds,
        quantiles,
        axis=1,
        method="nearest",
    )

    # Reshape quantiles so broadcasting works with x and targets.
    quantiles = np.expand_dims(
        quantiles,
        axis=list(range(1, len(preds.shape))),
    )

    # Quantile/CRPS loss summed across the forecast horizon T.
    loss = 2 * np.sum(
        np.abs(
            (x - targets)
            * ((targets <= x) - quantiles)
        ),
        axis=2,
    )

    # IMPORTANT:
    # This normalization is the same type of normalization
    # used in the Darts evaluation.
    #
    # Therefore, this is NOT the temporary raw NF CRPS
    # value (~21.38) that we calculated earlier.
    return loss.mean() / np.abs(targets).sum(axis=1).mean()


def crps_sum_like_theirs(
    preds,
    targets,
    quantiles=(np.arange(20) / 20.0)[1:],
):
    """
    Calculate CRPS after summing predictions across
    all Solar nodes.

    Parameters
    ----------
    preds : np.ndarray
        Shape:
            (B, N, T, D)

        B = forecast windows
        N = stochastic samples
        T = forecast horizon
        D = Solar nodes

    targets : np.ndarray
        Shape:
            (B, T, D)
    """

    # Sum predictions across all 137 Solar nodes.
    #
    # (B, N, T, D)
    #        ↓ sum over D
    # (B, N, T)
    preds_sum = preds.sum(axis=-1)

    # Sum actual values across all Solar nodes.
    #
    # (B, T, D)
    #        ↓ sum over D
    # (B, T)
    targets_sum = targets.sum(axis=-1)

    return crps(
        preds_sum,
        targets_sum,
        quantiles=quantiles,
    )


# ------------------------------------------------------------
# IMPORTANT:
# We are NOT calculating NF CRPS in this cell yet.
#
# First we need to confirm that NF + Engression is producing
# genuinely different stochastic trajectories.
#
# Expected first sanity-check shape:
#
#     (7, 24, 137)
#
# Then, for the final experiment:
#
#     (100, 24, 137)
#
# Once raw samples are confirmed, we will build the
# NeuralForecast-specific evaluation function below this cell.
# ------------------------------------------------------------

## Model

In [19]:
# Change ONLY this import when testing a different NeuralForecast
# Engression model.
#
# Examples:
# EnxLSTM              -> enxlstm
# EnPatchTST           -> enpatchtst
# EnAutoformer         -> enautoformer
# EnInformer           -> eninformer
# EnFEDformer          -> enfedformer
# EniTransformer       -> enitransformer
# EnTimeXer            -> entimexer
# EnTimesNet           -> entimesnet
# EnVanillaTransformer -> envanillatransformer
# EnTSMixerx           -> entsmixerx
# EnMLP                -> enmlp
# EnMLPMultivariate    -> enmlpmultivariate
# EnKAN                -> enkan
# EnBiTCN              -> enbitcn
# EnXLinear            -> enxlinear
#
# Keep the rest of the notebook unchanged when switching models.

from engressionts.models.neuralforecast.enxlstm import EnxLSTM

In [20]:
import inspect
from engressionts.base.base_engression import NFEngressionBaseModel

print(
    "_last_raw_samples" in inspect.getsource(
        NFEngressionBaseModel._predict_step_direct_batch
    )
)

print(
    "noise_layer.train()" in inspect.getsource(
        NFEngressionBaseModel._predict_step_direct_batch
    )
)


True
True


In [21]:
# NeuralForecast uses train_df directly.
#
# train_df format:
# unique_id | ds | y
#
# No Darts TimeSeries, scaler, or past_covariates are required.
# The model will use input_size=24 to take the previous 24 timesteps
# as context and h=24 to forecast the next 24 timesteps.

print("Training data shape:", train_df.shape)
print("Number of series/nodes:", train_df["unique_id"].nunique())
print("Columns:", train_df.columns.tolist())

Training data shape: (960233, 3)
Number of series/nodes: 137
Columns: ['unique_id', 'ds', 'y']


In [22]:
import logging
import lightning.pytorch as pl
# Lightning 2.x
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

# Older Lightning versions
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [23]:
import logging
import lightning.pytorch as pl
import time

In [24]:
# train_pc = train_pc.astype(np.float32)

In [24]:
# Check the NeuralForecast training data.
print("dtype:", train_df["y"].dtype)
print("shape:", train_df.shape)
print("number of nodes:", train_df["unique_id"].nunique())

dtype: float64
shape: (960233, 3)
number of nodes: 137


In [25]:
# print(train_y_sc.dtype)
# print(train_pc.dtype)

In [ ]:
SEED = 42

# Reproducibility
deterministic.init_all(SEED)
pl.seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)


# ============================================================
# NeuralForecast Engression model
# ============================================================
#
# When testing another NeuralForecast Engression model:
# 1. Change the model import in the import cell.
# 2. Change ONLY this model initialization block according
#    to that model's constructor.
#
# Common settings:
#   h=24          -> forecast next 24 timesteps
#   input_size=24 -> use previous 24 timesteps as context
#   num_samples=7 -> Engression samples used during training
#
# For the final probabilistic forecast, we can request/use
# more samples during prediction if supported by the wrapper.

model = EnxLSTM(
    h=24,
    input_size=24,

    noise_std=0.5632025703091124,
    noise_type="uniform",
    num_samples=7,

    learning_rate=0.0003992005679645662,
    batch_size=64,

    max_steps=3270,

    random_seed=SEED,

)


# ============================================================
# Train using NeuralForecast
# ============================================================

nf = NeuralForecast(
    models=[model],
    freq=ds.metadata.freq,
)

start = time.time()

nf.fit(
    df=train_df,
)

end = time.time()

print("Training time:", end - start)

Seed set to 42


Seed set to 42


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\Anusha\engression\engression-ts\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Training time: 960.9297549724579


: 

In [36]:
# ============================================================
# EnxLSTM prediction + raw stochastic trajectory check
# ============================================================

import torch
import numpy as np

# ------------------------------------------------------------
# 1. Clear raw samples from the ACTUAL executing NF model
# ------------------------------------------------------------

if hasattr(nf.models[0], "_last_raw_samples"):
    del nf.models[0]._last_raw_samples


# ------------------------------------------------------------
# 2. Run prediction ONCE
# ------------------------------------------------------------

forecast_df = nf.predict(
    df=train_df
)


# ------------------------------------------------------------
# 3. Retrieve raw samples from the executing NF model
# ------------------------------------------------------------

stored_batches = nf.models[0]._last_raw_samples

print("Number of stored sample batches:", len(stored_batches))


# ------------------------------------------------------------
# 4. Show batch shapes
# ------------------------------------------------------------

for i, batch in enumerate(stored_batches):
    print(
        f"Batch {i} shape:",
        tuple(batch.shape)
    )


# ------------------------------------------------------------
# 5. Combine the 3 batches
#
# 64 + 64 + 9 = 137 nodes
#
# Each batch:
# (nodes, 7 samples, 24 horizon, 1)
#
# Result:
# (137, 7, 24)
# ------------------------------------------------------------

all_samples_tensor = torch.cat(
    stored_batches,
    dim=0
).squeeze(-1)

print(
    "\nCombined raw tensor shape:",
    tuple(all_samples_tensor.shape)
)


# ------------------------------------------------------------
# 6. Convert to:
#
# (samples, horizon, nodes)
#
# = (7, 24, 137)
# ------------------------------------------------------------

raw_samples = (
    all_samples_tensor
    .permute(1, 2, 0)
    .detach()
    .cpu()
    .numpy()
)

print(
    "Raw stochastic samples shape:",
    raw_samples.shape
)


# ------------------------------------------------------------
# 7. Verify stochasticity
# ------------------------------------------------------------

diff_01 = np.abs(
    raw_samples[0] - raw_samples[1]
).sum()

diff_12 = np.abs(
    raw_samples[1] - raw_samples[2]
).sum()

print(
    "\nDifference between sample 0 and sample 1:",
    diff_01
)

print(
    "Difference between sample 1 and sample 2:",
    diff_12
)

assert diff_01 > 1e-4
assert diff_12 > 1e-4

print(
    "Stochastic sanity check PASSED."
)


# ------------------------------------------------------------
# 8. Verify raw samples reproduce forecast_df median
# ------------------------------------------------------------

raw_median_node_0 = np.median(
    raw_samples,
    axis=0
)[:, 0]

df_median_node_0 = (
    forecast_df[
        forecast_df["unique_id"] == "node_0"
    ]
    .sort_values("ds")["EnxLSTM-median"]
    .to_numpy()
)

max_abs_diff = np.abs(
    raw_median_node_0 - df_median_node_0
).max()

print(
    "\nMaximum absolute difference between "
    "raw-sample median and forecast_df median:",
    max_abs_diff
)

assert max_abs_diff < 1e-4

print(
    "Median verification PASSED."
)


# ------------------------------------------------------------
# 9. Final forecast information
# ------------------------------------------------------------

print("\nForecast shape:", forecast_df.shape)
print(
    "Forecast columns:",
    forecast_df.columns.tolist()
)

c:\Users\Anusha\engression\engression-ts\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

c:\Users\Anusha\engression\engression-ts\.venv\Lib\site-packages\neuralforecast\common\_scalers.py:30: UserWarning: median CUDA with indices output does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  x_median, _ = x_nan.nanmedian(dim=dim, keepdim=keepdim)


Number of stored sample batches: 3
Batch 0 shape: (64, 7, 24, 1)
Batch 1 shape: (64, 7, 24, 1)
Batch 2 shape: (9, 7, 24, 1)

Combined raw tensor shape: (137, 7, 24)
Raw stochastic samples shape: (7, 24, 137)

Difference between sample 0 and sample 1: 1920.2677
Difference between sample 1 and sample 2: 2024.2423
Stochastic sanity check PASSED.

Maximum absolute difference between raw-sample median and forecast_df median: 0.0
Median verification PASSED.

Forecast shape: (3288, 7)
Forecast columns: ['unique_id', 'ds', 'EnxLSTM-median', 'EnxLSTM-lo-90', 'EnxLSTM-lo-80', 'EnxLSTM-hi-80', 'EnxLSTM-hi-90']


In [37]:
# ============================================================
# Extract raw stochastic samples from the actual NF model
# ============================================================

nf_model = nf.models[0]

print("Number of stored batches:", len(nf_model._last_raw_samples))

for i, batch in enumerate(nf_model._last_raw_samples):
    print(f"Batch {i} shape:", tuple(batch.shape))

Number of stored batches: 3
Batch 0 shape: (64, 7, 24, 1)
Batch 1 shape: (64, 7, 24, 1)
Batch 2 shape: (9, 7, 24, 1)


In [38]:
nf_model = nf.models[0]

samples = nf_model._last_raw_samples

print("Number of batches:", len(samples))

for i, batch in enumerate(samples):
    print(
        f"Batch {i}: "
        f"shape={tuple(batch.shape)}, "
        f"min={batch.min().item():.6f}, "
        f"max={batch.max().item():.6f}, "
        f"mean={batch.mean().item():.6f}"
    )

Number of batches: 3
Batch 0: shape=(64, 7, 24, 1), min=-6.419043, max=20.601528, mean=-0.256907
Batch 1: shape=(64, 7, 24, 1), min=-7.143461, max=18.477991, mean=-0.261560
Batch 2: shape=(9, 7, 24, 1), min=-2.726041, max=7.871963, mean=-0.248695


In [39]:
# Check whether the 7 stochastic samples are actually different

batch = nf.models[0]._last_raw_samples[0].squeeze(-1)

print("Batch shape:", batch.shape)
# Expected: (64, 7, 24)

diff_01 = torch.abs(
    batch[:, 0, :] - batch[:, 1, :]
).sum()

diff_12 = torch.abs(
    batch[:, 1, :] - batch[:, 2, :]
).sum()

diff_06 = torch.abs(
    batch[:, 0, :] - batch[:, 6, :]
).sum()

print("Sample 0 vs Sample 1:", diff_01.item())
print("Sample 1 vs Sample 2:", diff_12.item())
print("Sample 0 vs Sample 6:", diff_06.item())

Batch shape: torch.Size([64, 7, 24])
Sample 0 vs Sample 1: 842.3464965820312
Sample 1 vs Sample 2: 875.656494140625
Sample 0 vs Sample 6: 863.3816528320312


In [40]:
nf_model = nf.models[0]

print("Number of raw sample batches:", len(nf_model._last_raw_samples))

for i, batch in enumerate(nf_model._last_raw_samples):
    print(
        f"Batch {i}: "
        f"shape={tuple(batch.shape)}, "
        f"mean={batch.mean().item():.4f}"
    )

print("\nForecast rows:", len(forecast_df))
print("Forecast rows per node:")
print(forecast_df.groupby("unique_id").size().value_counts())

Number of raw sample batches: 3
Batch 0: shape=(64, 7, 24, 1), mean=-0.2569
Batch 1: shape=(64, 7, 24, 1), mean=-0.2616
Batch 2: shape=(9, 7, 24, 1), mean=-0.2487

Forecast rows: 3288
Forecast rows per node:
24    137
Name: count, dtype: int64


In [42]:
# # Inspect the first sample from each of the 4 groups.
# # We will compare them with the corresponding forecast_df values.

# samples = nf.models[0]._last_raw_samples

# for group in range(4):
#     b0 = samples[group * 3]
    
#     # First node, first stochastic sample, first forecast timestep
#     value = b0[0, 0, 0, 0].item()
    
#     print(
#         f"Group {group + 1}: "
#         f"first batch shape={tuple(b0.shape)}, "
#         f"first prediction={value:.6f}"
#     )

# print("\nForecast_df first 10 predictions:")
# print(
#     forecast_df[
#         ["unique_id", "ds", "EnxLSTM-median"]
#     ].head(10)
# )

In [44]:
# # ============================================================
# # Verify that forecast_df median comes from the raw samples
# # ============================================================

# samples = nf.models[0]._last_raw_samples

# # Take the first group (batches 0, 1, 2)
# # and reconstruct its 137 nodes.

# group_1 = torch.cat(
#     samples[0:3],
#     dim=0
# ).squeeze(-1)

# print("Group 1 shape:", tuple(group_1.shape))
# # Expected:
# # (137, 7, 24)

# # Calculate the median across the 7 stochastic samples.
# group_1_median = torch.median(
#     group_1,
#     dim=1
# ).values

# print(
#     "Group 1 median shape:",
#     tuple(group_1_median.shape)
# )

# # Compare the first node's 24 predictions
# # with forecast_df's node_0 predictions.

# raw_median_node0 = group_1_median[0].detach().cpu().numpy()

# nf_median_node0 = (
#     forecast_df[
#         forecast_df["unique_id"] == "node_0"
#     ]["EnxLSTM-median"]
#     .to_numpy()
# )

# print("\nRaw-sample median for node_0:")
# print(raw_median_node0)

# print("\nforecast_df median for node_0:")
# print(nf_median_node0)

# print(
#     "\nMaximum absolute difference:",
#     np.max(
#         np.abs(
#             raw_median_node0 - nf_median_node0
#         )
#     )
# )

In [45]:
# ============================================================
# Inspect NeuralForecast prediction output
# ============================================================

print("forecast_df shape:", forecast_df.shape)

print("forecast_df columns:")
print(forecast_df.columns.tolist())

print("\nNumber of timesteps:")
print(forecast_df["ds"].nunique())

print("Number of nodes:")
print(forecast_df["unique_id"].nunique())

print("\nForecast rows per node:")
print(
    forecast_df.groupby("unique_id").size().value_counts()
)

forecast_df shape: (3288, 7)
forecast_df columns:
['unique_id', 'ds', 'EnxLSTM-median', 'EnxLSTM-lo-90', 'EnxLSTM-lo-80', 'EnxLSTM-hi-80', 'EnxLSTM-hi-90']

Number of timesteps:
24
Number of nodes:
137

Forecast rows per node:
24    137
Name: count, dtype: int64


In [46]:
# ============================================================
# NeuralForecast + Engression inference status
# ============================================================

print("Model ready for NeuralForecast inference.")
print("Model:", type(model).__name__)

print("Forecast horizon:", model.h)
print("Input size:", model.input_size)

print("Training samples per forecast:", model.num_samples)

# Raw stochastic samples captured from the ACTUAL
# NeuralForecast model used during prediction.
raw_samples = (
    torch.cat(
        nf.models[0]._last_raw_samples,
        dim=0
    )
    .squeeze(-1)
    .permute(1, 2, 0)
    .detach()
    .cpu()
    .numpy()
)

print("\nRaw stochastic samples shape:", raw_samples.shape)

print(
    "Expected format:",
    "(samples, horizon, nodes)"
)

print(
    "Samples:", raw_samples.shape[0],
    "| Horizon:", raw_samples.shape[1],
    "| Nodes:", raw_samples.shape[2]
)

Model ready for NeuralForecast inference.
Model: EnxLSTM
Forecast horizon: 24
Input size: 24
Training samples per forecast: 7

Raw stochastic samples shape: (7, 24, 137)
Expected format: (samples, horizon, nodes)
Samples: 7 | Horizon: 24 | Nodes: 137


In [47]:
# ============================================================
# NeuralForecast CRPS from RAW STOCHASTIC SAMPLES
# ============================================================
#
# raw_samples shape:
#
#     (N, T, D)
#
# N = stochastic samples
# T = forecast horizon
# D = Solar nodes
#
# Current test:
#
#     (7, 24, 137)
#
# Final evaluation:
#
#     (100, 24, 137)
#
# We use the same quantile-based CRPS formulation as the
# Darts/Engression evaluation for comparability.
# ============================================================

import numpy as np
import pandas as pd


def crps_from_samples(
    preds,
    targets,
    quantiles=(np.arange(20) / 20.0)[1:],
):
    """
    Calculate normalized CRPS from stochastic samples.

    Parameters
    ----------
    preds : np.ndarray
        Shape:
            (B, N, T, D)

        B = forecast windows
        N = stochastic samples
        T = forecast horizon
        D = Solar nodes

    targets : np.ndarray
        Shape:
            (B, T, D)

    Returns
    -------
    float
        Normalized CRPS.
    """

    # --------------------------------------------------------
    # Convert stochastic samples to quantiles.
    #
    # (B, N, T, D)
    #        ↓
    # (Q, B, T, D)
    # --------------------------------------------------------

    x = np.quantile(
        preds,
        quantiles,
        axis=1,
        method="nearest",
    )

    # Make quantiles broadcast correctly:
    #
    # (Q,)
    #   ↓
    # (Q, 1, 1, 1)

    quantiles = np.expand_dims(
        quantiles,
        axis=list(range(1, len(preds.shape))),
    )

    # --------------------------------------------------------
    # Quantile CRPS
    # --------------------------------------------------------

    loss = 2 * np.sum(
        np.abs(
            (x - targets)
            * ((targets <= x) - quantiles)
        ),
        axis=2,
    )

    # Same normalization used in the Darts evaluation.
    return (
        loss.mean()
        / np.abs(targets).sum(axis=1).mean()
    )


def get_nf_crps_from_samples(
    raw_samples,
    test_df,
    forecast_df,
    pred_len=24,
):
    """
    Calculate NF CRPS from the raw stochastic samples.

    raw_samples:
        (N, T, D)

        N = stochastic samples
        T = 24 forecast timesteps
        D = 137 Solar nodes

    test_df:
        NeuralForecast long-format Solar test data.

    forecast_df:
        NeuralForecast forecast output.
    """

    # --------------------------------------------------------
    # Check raw sample dimensions
    # --------------------------------------------------------

    if raw_samples.ndim != 3:
        raise ValueError(
            f"Expected raw_samples to have 3 dimensions "
            f"(N, T, D), got {raw_samples.shape}"
        )

    num_samples, horizon, num_nodes = raw_samples.shape

    if horizon != pred_len:
        raise ValueError(
            f"Expected horizon={pred_len}, "
            f"but got {horizon}"
        )

    # --------------------------------------------------------
    # Get the node ordering from forecast_df.
    #
    # NeuralForecast gives:
    #
    # node_0
    # node_1
    # ...
    # node_136
    #
    # raw_samples follows the same ordering that we verified
    # against forecast_df.
    # --------------------------------------------------------

    node_order = (
        forecast_df["unique_id"]
        .drop_duplicates()
        .tolist()
    )

    if len(node_order) != num_nodes:
        raise ValueError(
            f"Raw samples contain {num_nodes} nodes, "
            f"but forecast_df contains {len(node_order)} nodes."
        )

    # --------------------------------------------------------
    # Extract the 24 actual observations corresponding to
    # the forecast timestamps.
    # --------------------------------------------------------

    forecast_times = (
        forecast_df[
            ["unique_id", "ds"]
        ]
        .drop_duplicates()
    )

    actuals = test_df.merge(
        forecast_times,
        on=["unique_id", "ds"],
        how="inner",
    )

    # --------------------------------------------------------
    # Make sure every node has exactly 24 actual values.
    # --------------------------------------------------------

    counts = actuals.groupby("unique_id").size()

    if not (counts == pred_len).all():
        raise ValueError(
            "Some Solar nodes do not have exactly "
            f"{pred_len} matching actual observations."
        )

    # --------------------------------------------------------
    # Put actuals into:
    #
    # (nodes, horizon)
    # --------------------------------------------------------

    actuals = actuals.sort_values(
        ["unique_id", "ds"]
    )

    targets = np.stack(
        [
            actuals[
                actuals["unique_id"] == node
            ]["y"].to_numpy(dtype=np.float64)
            for node in node_order
        ],
        axis=0,
    )

    # targets:
    #
    # (137, 24)
    #
    # Convert to:
    #
    # (1, 24, 137)

    targets = targets.T[None, :, :]

    # --------------------------------------------------------
    # Convert raw samples:
    #
    # (N, T, D)
    #
    # to:
    #
    # (B, N, T, D)
    #
    # B = 1 forecast window
    # --------------------------------------------------------

    preds = raw_samples[None, :, :, :]

    print("Predictions shape:", preds.shape)
    print("Targets shape:", targets.shape)

    # --------------------------------------------------------
    # Calculate CRPS.
    # --------------------------------------------------------

    score = crps_from_samples(
        preds=preds,
        targets=targets,
    )

    return score


# ============================================================
# Evaluate current 7-sample EnxLSTM forecast
# ============================================================

nf_crps = get_nf_crps_from_samples(
    raw_samples=raw_samples,
    test_df=test_df,
    forecast_df=forecast_df,
    pred_len=24,
)

print("\nNF + Engression CRPS:", nf_crps)

Predictions shape: (1, 7, 24, 137)
Targets shape: (1, 24, 137)

NF + Engression CRPS: 1.0154308295082592


## Optimization

In [ ]:
import pickle
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
import numpy as np

# --- 1. Search Space ---
search_space = {
    "std": hp.uniform("std", 0.5, 3.0),
    "engression_m": hp.quniform("engression_m", 2, 8, 2),
    "learning_rate": hp.qloguniform(
        "learning_rate", np.log(1e-5), np.log(1e-2), 1e-5
    ),
    "batch_size": hp.quniform("batch_size", 32, 128, 32),
}

# --- 2. Objective Function ---
def objective(params):

    SEED = 42
    deterministic.init_all(SEED)
    pl.seed_everything(SEED, workers=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

    try:
        model = EnHiTSModel(
            input_chunk_length=24,
            output_chunk_length=24,
            noise_std=params["std"],
            num_samples_engression=int(params["engression_m"]),
            optimizer_kwargs={"lr": params["learning_rate"]},
            batch_size=int(params["batch_size"]),
            random_state=SEED,
            n_epochs=30,
            noise_dist="uniform",
        )

        model.fit(
            train_y_sc,
            past_covariates=train_pc,
            verbose=True,
            dataloader_kwargs={"num_workers": 0},
        )

        crps_list = []
        for i in range(10):
            crps_list.append(
                get_crps(
                    model,
                    test_windows,
                    y_scaler,
                    seed=SEED + i,
                )
            )

        mean_crps = np.mean(crps_list)
        std_crps = np.std(crps_list)

        print(f"Got Result {mean_crps} +/- {std_crps} for {params}")

        return {
            "loss": mean_crps,
            "status": STATUS_OK,
            "crps_list": crps_list,
        }

    except Exception as e:
        print(f"Trial failed: {e}")
        return {"loss": 1e6, "status": STATUS_OK}


# --- 3. Hyperopt Loop ---
trials_step = 5
max_trials = 50
trials_file = "hyperopt_trials_enhits.pkl"

try:
    with open(trials_file, "rb") as f:
        trials = pickle.load(f)
    print(f"Found existing trials. Resuming from {len(trials.trials)} runs.")
except:
    trials = Trials()

for i in range(len(trials.trials) + trials_step, max_trials + trials_step, trials_step):
    best = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=i,
        trials=trials,
        rstate=np.random.default_rng(SEED),
    )

    with open(trials_file, "wb") as f:
        pickle.dump(trials, f)

    print(f"Checkpoint saved at {i} trials. Best so far: {best}")

ModuleNotFoundError: No module named 'hyperopt'